In [ ]:
# Config
import os
import io
import re
import json
import time
import base64
import hmac
import hashlib
from datetime import datetime, timezone
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import soundfile as sf
import websocket
from jiwer import wer
from math import gcd
from scipy.signal import resample_poly
from datasets import load_dataset, Audio
from dotenv import load_dotenv, find_dotenv
from google.cloud import speech_v2
from google.api_core.client_options import ClientOptions

load_dotenv(find_dotenv(usecwd=True), override=False)

# Dataset + runtime config
DATASET_ID = "capleaf/viVoice"
SPLIT = "train"
LIMIT = 100
LANGUAGE_CODE = "vi-VN"
OUTPUT_DIR = "output/_compare_pipeline"

# Model config
GOOGLE_MODEL = "latest_short"
IFLYTEK_LANGUAGE = "vi_VN"
MAX_RETRIES = 2

# Credentials/env
HF_TOKEN = os.environ.get("HF_TOKEN")
GOOGLE_PROJECT_ID = os.environ.get("GOOGLE_PROJECT_ID")
GOOGLE_LOCATION = os.environ.get("GOOGLE_LOCATION", "us")
IFLYTEK_APPID = os.environ.get("IFLYTEK_APPID")
IFLYTEK_APIKEY = os.environ.get("IFLYTEK_APIKEY")
IFLYTEK_SECRET = os.environ.get("IFLYTEK_SECRET")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Dataset:", DATASET_ID, "| split:", SPLIT, "| limit:", LIMIT)
print("Google V2 enabled:", bool(GOOGLE_PROJECT_ID))
print("iFlytek enabled:", bool(IFLYTEK_APPID and IFLYTEK_APIKEY and IFLYTEK_SECRET))

Dataset: capleaf/viVoice | split: train | limit: 100
Google V2 enabled: True
iFlytek enabled: True


In [ ]:
# Helpers
def normalize_text(text):
    text = "" if text is None else str(text)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def get_duration_sec(audio_bytes):
    try:
        with io.BytesIO(audio_bytes) as bio:
            x, sr = sf.read(bio, dtype="float32")
        return float(len(x) / sr) if sr > 0 else 0.0
    except Exception:
        return 0.0


def to_pcm16le(audio_bytes, target_sr=16000):
    with io.BytesIO(audio_bytes) as bio:
        x, sr = sf.read(bio, dtype="float32")
    if x.ndim == 2:
        x = x.mean(axis=1)
    if sr != target_sr:
        g = gcd(sr, target_sr)
        x = resample_poly(x, target_sr // g, sr // g).astype(np.float32)
    x16 = (np.clip(x, -1.0, 1.0) * 32767.0).astype(np.int16)
    return x16.tobytes()


def build_iflytek_url(appkey, secret, host="iat-api-sg.xf-yun.com", path="/v2/iat"):
    date = datetime.now(timezone.utc).strftime("%a, %d %b %Y %H:%M:%S GMT")
    sign_origin = f"host: {host}\ndate: {date}\nGET {path} HTTP/1.1"
    sign_sha = hmac.new(secret.encode(), sign_origin.encode(), digestmod=hashlib.sha256).digest()
    sign_b64 = base64.b64encode(sign_sha).decode()
    auth_origin = (
        f'api_key="{appkey}", algorithm="hmac-sha256", '
        f'headers="host date request-line", signature="{sign_b64}"'
    )
    auth = base64.b64encode(auth_origin.encode()).decode()
    return f"wss://{host}{path}?" + urlencode({"authorization": auth, "date": date, "host": host})


def run_iflytek_word_by_word(pcm_bytes, appid, appkey, secret, language="vi_VN"):
    url = build_iflytek_url(appkey, secret)
    done = {"ok": False, "error": ""}
    words = []

    common = {"app_id": appid}
    business = {"domain": "iat", "language": language, "vinfo": 1, "vad_eos": 5000}

    def on_message(ws, message):
        resp = json.loads(message)
        code = resp.get("code", -1)
        if code != 0:
            done["ok"] = False
            done["error"] = f"{resp.get('message')} (code={code})"
            ws.close()
            return

        data = resp.get("data", {})
        result = data.get("result", {})
        for seg in result.get("ws", []):
            cands = seg.get("cw", [])
            token = cands[0].get("w", "") if cands else ""
            if token:
                words.append({
                    "word": token,
                    "wb": seg.get("wb"),
                    "we": seg.get("we"),
                })

        if data.get("status") == 2:
            done["ok"] = True
            ws.close()

    def on_error(ws, error):
        done["ok"] = False
        done["error"] = str(error)
        ws.close()

    def on_open(ws):
        frame_size = 8000
        interval = 0.04
        for i in range(0, len(pcm_bytes), frame_size):
            buf = pcm_bytes[i:i + frame_size]
            if not buf:
                break
            if i == 0:
                payload = {
                    "common": common,
                    "business": business,
                    "data": {
                        "status": 0,
                        "format": "audio/L16;rate=16000",
                        "audio": base64.b64encode(buf).decode(),
                        "encoding": "raw",
                    },
                }
            else:
                status = 2 if i + frame_size >= len(pcm_bytes) else 1
                payload = {"data": {"status": status, "audio": base64.b64encode(buf).decode()}}
            ws.send(json.dumps(payload))
            time.sleep(interval)

    ws = websocket.WebSocketApp(url, on_open=on_open, on_message=on_message, on_error=on_error)
    ws.run_forever()

    if not done["ok"]:
        raise RuntimeError(done["error"] or "iFlytek request failed")

    transcript = "".join([w["word"] for w in words])
    return transcript, words


def run_google_v2(audio_bytes, client, project_id, location, model, language_code):
    config = speech_v2.RecognitionConfig(
        auto_decoding_config=speech_v2.AutoDetectDecodingConfig(),
        model=model,
        language_codes=[language_code],
        features=speech_v2.RecognitionFeatures(enable_word_time_offsets=True),
    )
    request = speech_v2.RecognizeRequest(
        recognizer=f"projects/{project_id}/locations/{location}/recognizers/_",
        config=config,
        content=audio_bytes,
    )
    resp = client.recognize(request=request)

    final_transcript = ""
    words = []
    for result in resp.results:
        if not result.alternatives:
            continue
        alt = result.alternatives[0]
        if alt.transcript:
            final_transcript = alt.transcript
        for w in alt.words:
            words.append({
                "word": w.word,
                "start": w.start_offset.total_seconds(),
                "end": w.end_offset.total_seconds(),
            })

    return final_transcript, words

In [ ]:
# Run unified compare pipeline
start_all = time.perf_counter()

# Init models
google_client = None
if GOOGLE_PROJECT_ID:
    google_client = speech_v2.SpeechClient(
        client_options=ClientOptions(api_endpoint=f"{GOOGLE_LOCATION}-speech.googleapis.com")
    )

run_google = google_client is not None
run_iflytek = bool(IFLYTEK_APPID and IFLYTEK_APIKEY and IFLYTEK_SECRET)

if not run_google and not run_iflytek:
    raise RuntimeError("No model credentials found. Set Google and/or iFlytek env vars.")

# Load dataset
ds = load_dataset(DATASET_ID, split=SPLIT, streaming=True, token=HF_TOKEN)
ds = ds.cast_column("audio", Audio(decode=False))

rows = []
for idx, ex in enumerate(ds):
    if idx >= LIMIT:
        break

    gt = ex.get("text", "")
    audio_bytes = ex["audio"]["bytes"]
    duration_s = get_duration_sec(audio_bytes)

    # Google V2
    if run_google:
        err = ""
        transcript = ""
        words = []
        t0 = time.perf_counter()
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                transcript, words = run_google_v2(
                    audio_bytes,
                    google_client,
                    GOOGLE_PROJECT_ID,
                    GOOGLE_LOCATION,
                    GOOGLE_MODEL,
                    LANGUAGE_CODE,
                )
                err = ""
                break
            except Exception as e:
                err = str(e).replace("\n", " ")[:300]
                if attempt < MAX_RETRIES:
                    time.sleep(0.6 * attempt)
        elapsed = time.perf_counter() - t0
        gt_norm = normalize_text(gt)
        hyp_norm = normalize_text(transcript)
        wer_norm = wer(gt_norm, hyp_norm) if hyp_norm else 1.0

        rows.append({
            "index": idx,
            "model": "google_v2",
            "ground_truth": gt,
            "transcript": transcript,
            "ground_truth_norm": gt_norm,
            "transcript_norm": hyp_norm,
            "wer_norm": round(wer_norm, 3),
            "audio_duration_s": round(duration_s, 3),
            "processing_time_s": round(elapsed, 3),
            "rtf": round(elapsed / duration_s, 3) if duration_s > 0 else 0.0,
            "token_count": len(words),
            "has_native_timestamp": bool(words),
            "word_timestamps": json.dumps(words, ensure_ascii=False),
            "error": err,
        })

    # iFlytek
    if run_iflytek:
        err = ""
        transcript = ""
        tokens = []
        pcm = to_pcm16le(audio_bytes)
        t0 = time.perf_counter()
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                transcript, tokens = run_iflytek_word_by_word(
                    pcm,
                    IFLYTEK_APPID,
                    IFLYTEK_APIKEY,
                    IFLYTEK_SECRET,
                    IFLYTEK_LANGUAGE,
                )
                err = ""
                break
            except Exception as e:
                err = str(e).replace("\n", " ")[:300]
                if attempt < MAX_RETRIES:
                    time.sleep(0.6 * attempt)
        elapsed = time.perf_counter() - t0

        has_native = any(t.get("wb") is not None and t.get("we") is not None for t in tokens)
        if not has_native and tokens:
            weights = [max(1, len(str(t.get("word", "")).strip())) for t in tokens]
            total_w = float(sum(weights)) if sum(weights) > 0 else float(len(tokens))
            cur = 0.0
            for t, w in zip(tokens, weights):
                seg = duration_s * (w / total_w)
                t["est_start"] = cur
                cur += seg
                t["est_end"] = cur

        gt_norm = normalize_text(gt)
        hyp_norm = normalize_text(transcript)
        wer_norm = wer(gt_norm, hyp_norm) if hyp_norm else 1.0

        rows.append({
            "index": idx,
            "model": "iflytek",
            "ground_truth": gt,
            "transcript": transcript,
            "ground_truth_norm": gt_norm,
            "transcript_norm": hyp_norm,
            "wer_norm": round(wer_norm, 3),
            "audio_duration_s": round(duration_s, 3),
            "processing_time_s": round(elapsed, 3),
            "rtf": round(elapsed / duration_s, 3) if duration_s > 0 else 0.0,
            "token_count": len(tokens),
            "has_native_timestamp": has_native,
            "word_timestamps": json.dumps(tokens, ensure_ascii=False),
            "error": err,
        })

    print(f"[{idx+1:03d}/{LIMIT}] done")

result_df = pd.DataFrame(rows)

detail_path = os.path.join(OUTPUT_DIR, "compare_detailed.csv")
summary_path = os.path.join(OUTPUT_DIR, "compare_summary.csv")

result_df.to_csv(detail_path, index=False, encoding="utf-8-sig")

summary_df = (
    result_df
    .groupby("model", as_index=False)
    .agg(
        samples=("model", "count"),
        avg_wer_norm=("wer_norm", "mean"),
        median_wer_norm=("wer_norm", "median"),
        avg_rtf=("rtf", "mean"),
        error_count=("error", lambda s: int((s.fillna("").astype(str).str.strip() != "").sum())),
    )
    .sort_values("avg_wer_norm")
)
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\n=== Compare Summary ===")
display(summary_df)
print("Saved detail:", detail_path)
print("Saved summary:", summary_path)
print("Total time (s):", round(time.perf_counter() - start_all, 2))

[001/100] done
[002/100] done
[003/100] done
[004/100] done
[005/100] done
[006/100] done
[007/100] done
[008/100] done
[009/100] done
[010/100] done
[011/100] done
[012/100] done
[013/100] done
[014/100] done
[015/100] done
[016/100] done
[017/100] done
[018/100] done
[019/100] done
[020/100] done
[021/100] done
[022/100] done
[023/100] done
[024/100] done
[025/100] done
[026/100] done
[027/100] done
[028/100] done
[029/100] done
[030/100] done
[031/100] done
[032/100] done
[033/100] done
[034/100] done
[035/100] done
[036/100] done
[037/100] done
[038/100] done
[039/100] done
[040/100] done
[041/100] done
[042/100] done
[043/100] done
[044/100] done
[045/100] done
[046/100] done
[047/100] done
[048/100] done
[049/100] done
[050/100] done
[051/100] done
[052/100] done
[053/100] done
[054/100] done
[055/100] done
[056/100] done
[057/100] done
[058/100] done
[059/100] done
[060/100] done
[061/100] done
[062/100] done
[063/100] done
[064/100] done
[065/100] done
[066/100] done
[067/100] 

,model,samples,avg_wer_norm,median_wer_norm,avg_rtf,error_count
1,iflytek,100,0.11668,0.052,0.38462,0
0,google_v2,100,0.21861,0.091,0.52518,0


Saved detail: output/_compare_pipeline/compare_detailed.csv
Saved summary: output/_compare_pipeline/compare_summary.csv
Total time (s): 312.75


In [10]:
# Cell 4 - Quick inspection
show_n = 5

df = pd.read_csv(os.path.join(OUTPUT_DIR, "compare_detailed.csv"), index_col=False)

print("Rows:", len(df))
print("Models:", sorted(df["model"].unique().tolist()))

print("\nWorst samples by WER per model:")
for model in sorted(df["model"].unique().tolist()):
    print(f"\n--- {model} ---")
    view = df[df["model"] == model].sort_values("wer_norm", ascending=False).head(show_n)
    display(view[["index", "wer_norm", "rtf", "ground_truth", "transcript", "error"]])

Rows: 200
Models: ['google_v2', 'iflytek']

Worst samples by WER per model:

--- google_v2 ---


,index,wer_norm,rtf,ground_truth,transcript,error
128,64,1.0,0.216,Alfred Jarry 1873-1907 hợp những nhà văn đổi t...,professor,NaN
134,67,1.0,0.823,Chết rồi.,NaN,NaN
194,97,1.0,0.628,Khởi động.,NaN,NaN
14,7,1.0,1.045,Thì kết thúc này quen lắm.,NaN,NaN
188,94,1.0,0.379,Srimad Bhagavad Gita xuất hiện trong Mahabhara...,smartphone guitar,NaN



--- iflytek ---


,index,wer_norm,rtf,ground_truth,transcript,error
129,64,1.364,0.223,Alfred Jarry 1873-1907 hợp những nhà văn đổi t...,Alfred zari 1 ngàn 800 731 ngàn 900 lẻ 7 và t...,NaN
135,67,1.000,0.884,Chết rồi.,Cho tôi.,NaN
195,97,1.000,0.585,Khởi động.,Không.,NaN
79,39,0.429,0.250,TV có không hình hiển thị sắc nét nhờ độ phân ...,Tivi có khung hình hiển thị sắt thép nhờ độ p...,NaN
189,94,0.421,0.222,Srimad Bhagavad Gita xuất hiện trong Mahabhara...,Stremat bà gặp vasghita xuất hiện trong mahab...,NaN
